# Index spread moves

Pick an index (`HY47`, `HY42`, `XO44`, `IG45`, `MA40`, ...) and two dates. First it shows which names are missing data, then one row per index member with the spread, `upfront`, `all_upfront` and `risky_pv01` at both dates and the change in each, sorted by whichever change you pick (`SORT_BY`).

- Reads the published snapshot in `data/` only (no Databricks / Bloomberg / Excel, and this notebook never writes data). The first cell prints which snapshot it is (published date and last price date). The data is built and published by the owner with the scripts in `pricing/`.
- Spread is `conv_spread` in bp at the index maturity; `upfront` and `all_upfront` are in points. Dates snap to the last available date on or before the request, and an index has no data before its launch date.
- Names with no price on a date stay in the table with blank values and a `status` saying why: `missing` (no Databricks row, red). Rows shaded amber are wide names (`recal`), blue rows are re-priced from the other coupon (`recoupon`), purple rows come from the Markit add-in (`addin`). A `spread_override` from `overrides.csv` beats all of these.

In [1]:
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'analysis' / 'cdx_spreads.py').exists())     # project root, wherever the notebook is opened from
sys.path.insert(0, str(ROOT / 'analysis'))

import pandas as pd
import cdx_spreads as cs
from IPython.display import HTML, display

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 300)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.expand_frame_repr', False)

# no-wrap HTML table with fixed decimals, blanks for missing, red rows for names missing data
show = cs.show_table


print(', '.join(cs.load_index_list()['index_name']))
cs.print_data_as_of()          # which published snapshot (data/) this notebook is reading


IG47, IG46, IG45, IG44, IG43, IG42, IG41, IG40, IG39, IG38, HY47, HY46, HY45, HY44, HY43, HY42, HY41, HY40, HY39, HY38, XO46, XO45, XO44, XO43, XO42, XO41, XO40, XO39, XO38, XO37, MA46, MA45, MA44, MA43, MA42, MA41, MA40, MA39, MA38, MA37
data published 2026-10-26 08:34; prices through 2026-10-23


## Parameters

In [2]:
INDEX = 'HY47'
DATE_X = '2026-09-29'   # start
DATE_Y = '2026-10-23'   # end
SORT_BY = 'spread'      # which change to sort on: 'spread', 'upfront', 'all_upfront' or 'risky_pv01'
SORT_BY_ABS = False     # False: largest positive change on top (widest first for spreads). True: largest move in either direction first.

## 1. Names not priced
**Priced** means a curve point has a `par_spread`, or a `conv_spread` / `upfront` together with the coupon it is quoted at (`risky_pv01` is optional: `cds_pricer.py` derives it from a flat curve). A `conv_spread` of 0 is Markit's 'no spread solves' placeholder, not a price. The goal is for every name on every date to be priced.

Sources, in order: **Databricks** first, then the **Markit add-in** (`addin_fill.py`) for cells Databricks could not price, then any manual override. Every spread, upfront, `all_upfront` and `risky_pv01` below is **at the index coupon** (500 for HY/XO, 100 for IG/MA).

- **Red box:** names with no price on one of the dates, grouped by reason (*never priced in this index*, *conv=0*, *incomplete* stub row, *gap*).
- **Amber box:** wide names priced with a marked-down recovery (`recovery = 100 - all_upfront - 5`, flat 4% discounting): a clean comparable spread, but not Markit's.
- **Blue box:** names Markit quotes at the other coupon, re-priced to the index coupon with `cds_pricer.py`.
- **Purple box:** names priced from the Markit add-in because Databricks has no price (renamed or missing curves); half-year tenors are interpolated between annual nodes.

The table gives, for each flagged name, `days_with_data` of `days_in_index` (priced by either source), `first_data`, the coupon it was quoted at and the recovery used.

In [3]:
res = cs.spread_changes(INDEX, DATE_X, DATE_Y)
a = res.attrs
own, recoupon, recal, addin, unpriced = cs.priced_counts(res)
print(f"{INDEX}: conv_spread {a['date_x']:%Y-%m-%d} -> {a['date_y']:%Y-%m-%d} | index maturity {a['maturity']:%Y-%m-%d} | index coupon {a['index_coupon']} | "
      f"{len(res)} names: {own} priced by Databricks at the index coupon, {recoupon} re-priced from the other coupon, {recal} wide names at a marked-down recovery, "
      f"{addin} priced from the Markit add-in, {unpriced} not priced")
display(HTML(cs.missing_banner_html(res)))
show(cs.missing_report(res), numeric={'quoted coupon X': 0, 'quoted coupon Y': 0, 'recovery X (%)': 1, 'recovery Y (%)': 1})

HY47: conv_spread 2026-09-29 -> 2026-10-23 | index maturity 2031-12-20 | index coupon 500 | 100 names: 67 priced by Databricks at the index coupon, 22 re-priced from the other coupon, 4 wide names at a marked-down recovery, 7 priced from the Markit add-in, 0 not priced


member_name,Market_CDS_Ticker,what,status,days_with_data,days_in_index,first_data,quoted coupon X,quoted coupon Y,recovery X (%),recovery Y (%)
CSC Holdings LLC,CSCHLD,"Priced, wide name: conv_spread at a marked-down recovery",X recal; Y recal,20,20,2026-09-28,500,500,15.4,13.8
Hertz Corp/The,HTZ,"Priced, wide name: conv_spread at a marked-down recovery",X recal; Y recal,20,20,2026-09-28,500,500,14.4,12.1
Xerox Corp,XRX,"Priced, wide name: conv_spread at a marked-down recovery",X recal; Y recal,20,20,2026-09-28,100,100,26.3,21.2
iHeartCommunications Inc,IHRT,"Priced, wide name: conv_spread at a marked-down recovery",X recal; Y recal,20,20,2026-09-28,500,500,39.7,30.0
Ball Corp,BALL,"Priced, re-priced from the other coupon to the index coupon",X recoupon; Y recoupon,20,20,2026-09-28,100,100,,
Bath & Body Works Inc,BBWI,"Priced, re-priced from the other coupon to the index coupon",X recoupon; Y recoupon,20,20,2026-09-28,100,100,,
Discovery Global Holdings Inc,WBD,"Priced, re-priced from the other coupon to the index coupon",X recoupon; Y recoupon,20,20,2026-09-28,100,100,,
Domtar Corp,UFS,"Priced, re-priced from the other coupon to the index coupon",X recoupon; Y recoupon,20,20,2026-09-28,100,100,,
Gap Inc/The,GAP,"Priced, re-priced from the other coupon to the index coupon",X recoupon; Y recoupon,20,20,2026-09-28,100,100,,
Kohl's Corp,KSS,"Priced, re-priced from the other coupon to the index coupon",X recoupon; Y recoupon,20,20,2026-09-28,100,100,,


## 2. Largest moves
Sorted by the change chosen in `SORT_BY` (see `SORT_BY_ABS` above). Row color shows where a name's numbers came from: plain rows are white on black, tinted rows are black on a light tint.

| Row shading | Meaning |
|---|---|
| black (no tint) | priced by Databricks at the index coupon |
| blue | Markit quotes the name at the other coupon; re-priced to the index coupon with `cds_pricer` |
| purple | Databricks has no price on at least one date; taken from the Markit add-in |
| amber, bold | wide name: no conventional spread solves at 40% recovery, so it is priced at a marked-down recovery (recovery columns show which) |
| red, bold | no price on at least one of the two dates; blank spreads, sorted to the bottom |

In [4]:
print(f"median spread change {res['change'].median():.1f} bp | median pct change {res['pct_change'].median():.1f}%")

sort_col = {'spread': 'change', 'upfront': 'upfront_change', 'all_upfront': 'all_upfront_change', 'risky_pv01': 'risky_pv01_change'}[SORT_BY]
res = res.sort_values(sort_col, key=abs if SORT_BY_ABS else None, ascending=False, na_position='last').reset_index(drop=True)

dx, dy = f"{a['date_x']:%Y-%m-%d}", f"{a['date_y']:%Y-%m-%d}"
cols = {'conv_spread_x': f'spread {dx}', 'conv_spread_y': f'spread {dy}', 'change': 'spread change (bp)', 'pct_change': 'spread change (%)'}
for m in ('upfront', 'all_upfront', 'risky_pv01'):
    cols |= {f'{m}_x': f'{m} {dx}', f'{m}_y': f'{m} {dy}', f'{m}_change': f'{m} change'}
cols |= {'recovery_x': f'recovery used {dx} (%)', 'recovery_y': f'recovery used {dy} (%)'}   # only filled on recalibrated rows
view = res[['member_name', 'Market_CDS_Ticker', *cols, 'status']].rename(columns=cols)
places = {name: (2 if name.startswith('risky_pv01') else 1) for name in cols.values()}   # risky_pv01: 2 decimals, everything else 1
show(view, numeric=places, flag_col='status')   # red = missing data, amber = recalibrated

median spread change 0.6 bp | median pct change 0.6%


member_name,Market_CDS_Ticker,spread 2026-09-29,spread 2026-10-23,spread change (bp),spread change (%),upfront 2026-09-29,upfront 2026-10-23,upfront change,all_upfront 2026-09-29,all_upfront 2026-10-23,all_upfront change,risky_pv01 2026-09-29,risky_pv01 2026-10-23,risky_pv01 change,recovery used 2026-09-29 (%),recovery used 2026-10-23 (%),status
iHeartCommunications Inc,IHRT,2673.6,3734.6,1061.0,39.7,41.5,53.6,12.1,55.3,65.0,9.7,2.76,2.28,-0.48,39.7,30.0,X recal; Y recal
Staples Inc,SPLS,1791.7,2632.5,840.7,46.9,31.8,40.9,9.1,46.8,53.9,7.1,3.01,2.60,-0.41,,,ok
Xerox Corp,XRX,3910.9,4616.8,705.9,18.1,56.9,63.3,6.5,68.7,73.8,5.2,2.36,2.10,-0.26,26.3,21.2,X recal; Y recal
MPT Operating Partnership LP,MPW,1416.6,1830.1,413.5,29.2,25.5,32.2,6.7,40.1,45.3,5.1,2.93,2.62,-0.31,,,ok
Sabre Holdings Corp,SABHLD,834.3,1230.9,396.5,47.5,11.4,21.5,10.1,29.1,37.3,8.2,3.53,3.15,-0.38,,,ok
CoreWeave Inc,CRWV,837.6,1141.0,303.4,36.2,11.5,19.5,7.9,29.3,35.1,5.8,3.55,3.12,-0.43,,,ok
Beazer Homes USA Inc,BZH,271.6,544.0,272.4,100.3,-9.7,1.7,11.4,11.9,21.5,9.6,4.32,3.97,-0.35,,,ok
CSC Holdings LLC,CSCHLD,8804.3,9066.0,261.7,3.0,76.6,78.3,1.7,79.6,81.2,1.7,0.59,0.59,0.00,15.4,13.8,X recal; Y recal
Whirlpool Corp,WHR,534.7,775.9,241.2,45.1,1.3,9.6,8.2,21.3,28.1,6.8,3.99,3.70,-0.29,,,X recoupon; Y recoupon
Hertz Corp/The,HTZ,5478.1,5713.8,235.7,4.3,71.3,73.6,2.3,80.6,82.9,2.3,1.87,1.87,0.00,14.4,12.1,X recal; Y recal


## 3. Pricing coverage, all indices
How much of the store is priced, per index (cells = index x member x date). `priced` is Databricks by the agreed definition; `quoted at index coupon %` counts cells Markit quotes at the index's own coupon (the rest are re-priced). `filled from add-in` is how many of the `not priced` cells the Markit add-in prices, `priced % incl. add-in` the resulting coverage, and `still not priced` what is left. `conv=0 (priced via par)` counts wide-name rows that Markit gives no conv_spread but which still count as priced.

In [5]:
show(cs.pricing_coverage(), numeric={'priced %': 1, 'quoted at index coupon %': 1, 'priced % incl. add-in': 1})

index,cells,priced,priced %,quoted at index coupon %,not priced,filled from add-in,priced % incl. add-in,still not priced,no row,empty row,conv=0 (priced via par)
IG47,3125,3050,97.6,83.2,75,75,100.0,0,75,0,0
IG46,19500,19230,98.6,85.0,270,270,100.0,0,270,0,0
IG45,35625,34793,97.7,83.3,832,793,99.9,39,831,1,0
IG44,52125,51161,98.2,84.6,964,793,99.7,171,963,1,0
IG43,68250,66261,97.1,83.5,1989,1689,99.6,300,1987,2,0
IG42,84750,80265,94.7,81.9,4485,4053,99.5,432,4484,1,0
IG41,101000,93248,92.3,80.3,7752,7190,99.4,562,7751,1,0
IG40,117500,109058,92.8,80.8,8442,7462,99.2,980,8442,0,0
IG39,133625,123635,92.5,81.3,9990,8395,98.8,1595,9990,0,0
IG38,148875,136558,91.7,82.9,12317,10092,98.5,2225,12316,1,0


## 4. Index metrics
Built from each name's `all_upfront` (its present value of expected loss, in points; Markit's value from Databricks where priced there, else the Markit add-in). One row per index and date.

- **dispersion**: standard deviation of `all_upfront` across the index's priced names (unweighted, population std).
- **index_el**: index expected loss = sum of weight x `all_upfront` over the priced names, divided by the weight of all names live on that date (points).
- **top5_share / top10_share**: expected loss of the 5 / 10 widest names (largest `all_upfront`) x their weights, divided by the total over priced names.

Names that defaulted out of a series (from `teams.structured_credit_abhutra.credit_events`) stay in it at their original weight until `index_ex_date`, using their own prices (Databricks, else the Markit add-in), so earlier dates still include them. Unpriced names are left out of that date; `n_priced` and `priced_weight` show how much of the live index each number covers.

In [6]:
import index_metrics as im

met = im.index_metrics(INDEX).set_index('published_date')
two = met.loc[[a['date_x'], a['date_y']], ['dispersion', 'index_el', 'top5_share', 'top10_share', 'max_el', 'n_priced', 'n_members', 'priced_weight']].copy()
two.loc['change'] = two.iloc[1] - two.iloc[0]
for c in ('top5_share', 'top10_share', 'priced_weight'):
    two[c] = two[c] * 100                       # show as %
two = two.rename(columns={'top5_share': 'top5 share (%)', 'top10_share': 'top10 share (%)', 'priced_weight': 'priced weight (%)', 'index_el': 'index EL (pts)', 'max_el': 'widest (pts)'})
two.insert(0, 'date', [f"{a['date_x']:%Y-%m-%d}", f"{a['date_y']:%Y-%m-%d}", 'change'])
show(two, numeric={'dispersion': 2, 'index EL (pts)': 2, 'top5 share (%)': 1, 'top10 share (%)': 1, 'widest (pts)': 1, 'n_priced': 0, 'n_members': 0, 'priced weight (%)': 1})

date,dispersion,index EL (pts),top5 share (%),top10 share (%),widest (pts),n_priced,n_members,priced weight (%)
2026-09-29,14.83,14.43,23.3,35.9,80.6,100,100,100.0
2026-10-23,16.00,15.46,23.1,36.1,82.9,100,100,100.0
change,1.17,1.03,-0.1,0.1,2.3,0,0,0.0


The widest names on the end date, and what share of the index's expected loss each carries:

In [7]:
w = im.widest(INDEX, a['date_y'], 10)
w['weight'] = w['weight'] * 100
w['share_of_index_el'] = w['share_of_index_el'] * 100
w = w.rename(columns={'all_upfront': 'all_upfront (pts)', 'weight': 'weight (%)', 'el_contribution': 'EL contribution (pts)', 'share_of_index_el': 'share of index EL (%)'})
show(w, numeric={'all_upfront (pts)': 1, 'weight (%)': 2, 'EL contribution (pts)': 2, 'share of index EL (%)': 1})

member_name,market_cds_ticker,all_upfront (pts),weight (%),EL contribution (pts),share of index EL (%),source
Hertz Corp/The,HTZ,82.9,1.00,0.83,5.4,databricks
CSC Holdings LLC,CSCHLD,81.9,1.00,0.82,5.3,databricks
Xerox Corp,XRX,73.8,1.00,0.74,4.8,databricks
iHeartCommunications Inc,IHRT,65.1,1.00,0.65,4.2,databricks
Staples Inc,SPLS,53.9,1.00,0.54,3.5,databricks
Domtar Corp,UFS,47.5,1.00,0.47,3.1,databricks
MPT Operating Partnership LP,MPW,45.3,1.00,0.45,2.9,databricks
Sabre Holdings Corp,SABHLD,37.3,1.00,0.37,2.4,databricks
CoreWeave Inc,CRWV,35.1,1.00,0.35,2.3,databricks
Bausch Health Cos Inc,BHCCN,35.0,1.00,0.35,2.3,databricks
